# 02 — Cohort and serum-creatinine-defined outcome construction

Public source-only reproducibility notebook. Outputs and execution history were removed. No credentials, patient-level data, row-level predictions, or row-level SHAP values are included. Execution requires credentialed access to the eICU Collaborative Research Database and an authorized Google Cloud project.


In [ ]:
import os
from pathlib import Path

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
WORK_DATASET_NAME = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
SOURCE_DATASET = os.environ.get("EICU_SOURCE_DATASET", "physionet-data.eicu_crd")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")
OUTPUT_ROOT = os.environ.get("AKI_OUTPUT_ROOT", "/content/AKI_JCMC_V2_PUBLIC_RUN")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

print("Target dataset:", TARGET_DATASET)
print("Output root:", OUTPUT_ROOT)


# AKI V2 — Kohort ve Dinamik KDIGO Outcome Çalıştırıcısı v1.0

Bu notebook hasta düzeyindeki güvenli tabloları BigQuery projenizde oluşturur; yalnızca toplulaştırılmış CSV sonuçlarını Drive’a kaydeder.

In [ ]:
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
SOURCE_DATASET = "physionet-data.eicu_crd"
WORK_DATASET_NAME = globals().get("WORK_DATASET_NAME") or os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
BQ_LOCATION = "US"
DRIVE_OUTPUT_DIR = f"{OUTPUT_ROOT}/02_COHORT_OUTCOME_OUTPUTS"
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"


In [ ]:
!pip -q install google-cloud-bigquery pandas pyarrow openpyxl
from google.colab import auth, drive
from google.cloud import bigquery
import pandas as pd, os, json, zipfile, hashlib, datetime, pathlib
auth.authenticate_user()
drive.mount('/content/drive')
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print('BigQuery client ready:', PROJECT_ID)
print('Target dataset:', TARGET_DATASET)
print('Output directory:', DRIVE_OUTPUT_DIR)


In [ ]:
dataset = bigquery.Dataset(TARGET_DATASET)
dataset.location = BQ_LOCATION
client.create_dataset(dataset, exists_ok=True)
print('Dataset ready:', TARGET_DATASET)

def run_ddl(label, sql):
    print('Running:', label)
    client.query(sql).result()
    print('Completed:', label)

def run_aggregate(label, sql, filename):
    print('Running:', label)
    df = client.query(sql).to_dataframe()
    display(df.head(100))
    path = os.path.join(DRIVE_OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print('Saved:', path, 'rows=', len(df))
    return df


## 01_create_base_stays_v1.sql

In [ ]:
SQL_01 = f"""-- Persistent secure table: adult stays and identifier structure.
CREATE OR REPLACE TABLE `{TARGET_DATASET}.base_stays_v1` AS
WITH adults AS (
  SELECT
    patientUnitStayID,
    patientHealthSystemStayID,
    uniquePID,
    hospitalID,
    unitVisitNumber,
    hospitalAdmitOffset,
    unitDischargeOffset,
    LOWER(TRIM(unitDischargeStatus)) AS unit_discharge_status,
    LOWER(TRIM(hospitalDischargeStatus)) AS hospital_discharge_status,
    CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END AS age_num
  FROM `{SOURCE_DATASET}.patient`
  WHERE (CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END) >= 18
    AND unitDischargeOffset >= 720
),
patient_structure AS (
  -- Use the full patient table, not only eligible 12 h stays, to detect cross-hospital patients.
  SELECT
    uniquePID,
    COUNT(DISTINCT hospitalID) AS n_hospitals_for_patient,
    COUNT(DISTINCT patientHealthSystemStayID) AS n_hospital_stays_for_patient,
    COUNT(DISTINCT patientUnitStayID) AS n_unit_stays_for_patient
  FROM `{SOURCE_DATASET}.patient`
  GROUP BY uniquePID
),
ranked AS (
  SELECT
    a.*,
    s.n_hospitals_for_patient,
    s.n_hospital_stays_for_patient,
    s.n_unit_stays_for_patient,
    ROW_NUMBER() OVER (
      PARTITION BY a.patientHealthSystemStayID
      ORDER BY COALESCE(a.unitVisitNumber, 999999), a.patientUnitStayID
    ) AS eligible_unit_rank_in_hospstay
  FROM adults a
  JOIN patient_structure s USING (uniquePID)
)
SELECT * FROM ranked;
"""
run_ddl("01_create_base_stays_v1.sql", SQL_01)


## 02_create_creatinine_long_v1.sql

In [ ]:
SQL_02 = f"""-- Persistent secure table: one revised serum creatinine value per stay/offset.
CREATE OR REPLACE TABLE `{TARGET_DATASET}.creatinine_long_v1` AS
WITH raw AS (
  SELECT
    b.patientUnitStayID,
    b.patientHealthSystemStayID,
    b.uniquePID,
    b.hospitalID,
    b.hospitalAdmitOffset,
    b.unitDischargeOffset,
    l.labID,
    l.labResultOffset,
    l.labResultRevisedOffset,
    SAFE_CAST(l.labResult AS FLOAT64) AS creatinine,
    ROW_NUMBER() OVER (
      PARTITION BY l.patientUnitStayID, l.labResultOffset
      ORDER BY COALESCE(l.labResultRevisedOffset, l.labResultOffset) DESC, l.labID DESC
    ) AS revision_rank
  FROM `{TARGET_DATASET}.base_stays_v1` b
  JOIN `{SOURCE_DATASET}.lab` l USING (patientUnitStayID)
  WHERE b.eligible_unit_rank_in_hospstay = 1
    AND b.n_hospitals_for_patient = 1
    AND LOWER(TRIM(l.labName)) = 'creatinine'
    AND SAFE_CAST(l.labResult AS FLOAT64) BETWEEN 0.1 AND 30.0
    AND l.labResultOffset BETWEEN GREATEST(b.hospitalAdmitOffset, -1440)
                              AND LEAST(b.unitDischargeOffset, 4320)
)
SELECT * EXCEPT(revision_rank)
FROM raw
WHERE revision_rank = 1;
"""
run_ddl("02_create_creatinine_long_v1.sql", SQL_02)


## 03_create_renal_flags_v1.sql

In [ ]:
SQL_03 = f"""-- Persistent secure table: strict chronic dialysis/ESRD and RRT timing flags.
CREATE OR REPLACE TABLE `{TARGET_DATASET}.renal_flags_v1` AS
WITH base AS (
  SELECT *
  FROM `{TARGET_DATASET}.base_stays_v1`
  WHERE eligible_unit_rank_in_hospstay = 1
    AND n_hospitals_for_patient = 1
),
ph AS (
  SELECT
    patientUnitStayID,
    MAX(IF(
      REGEXP_CONTAINS(LOWER(CONCAT(COALESCE(pastHistoryPath,''),' ',COALESCE(pastHistoryValue,''),' ',COALESCE(pastHistoryValueText,''))),
        r'end.?stage renal|\besrd\b|dialysis.?dependent|chronic dialysis|maintenance dialysis|for chronic renal failure'),
      1, 0)) AS strict_chronic_ph,
    MAX(IF(
      REGEXP_CONTAINS(LOWER(CONCAT(COALESCE(pastHistoryPath,''),' ',COALESCE(pastHistoryValue,''),' ',COALESCE(pastHistoryValueText,''))),
        r'arteriovenous shunt|dialysis access|renal failure|chronic kidney'),
      1, 0)) AS broad_chronic_ph
  FROM `{SOURCE_DATASET}.pasthistory`
  WHERE pastHistoryOffset <= 720
  GROUP BY patientUnitStayID
),
dx AS (
  SELECT
    patientUnitStayID,
    MAX(IF(REGEXP_CONTAINS(LOWER(diagnosisString),
      r'end.?stage renal|\besrd\b|dialysis.?dependent|chronic dialysis|maintenance dialysis'),1,0)) AS strict_chronic_dx,
    MAX(IF(REGEXP_CONTAINS(LOWER(diagnosisString),
      r'chronic renal|chronic kidney|dialysis access|arteriovenous shunt'),1,0)) AS broad_chronic_dx
  FROM `{SOURCE_DATASET}.diagnosis`
  WHERE diagnosisOffset <= 720
  GROUP BY patientUnitStayID
),
tx AS (
  SELECT
    patientUnitStayID,
    MAX(IF(treatmentOffset <= 720 AND REGEXP_CONTAINS(LOWER(treatmentString),
      r'for chronic renal failure|chronic dialysis|maintenance dialysis|dialysis.?dependent'),1,0)) AS strict_chronic_tx,
    MAX(IF(treatmentOffset <= 720 AND REGEXP_CONTAINS(LOWER(treatmentString),
      r'dialysis access|arteriovenous shunt|for chronic renal failure'),1,0)) AS broad_chronic_tx,
    MIN(IF(
      treatmentOffset >= 0
      AND NOT REGEXP_CONTAINS(LOWER(treatmentString), r'for chronic renal failure|dialysis access|arteriovenous shunt|catheter')
      AND REGEXP_CONTAINS(LOWER(treatmentString),
        r'for acute renal failure|hemodialysis\|emergent|peritoneal dialysis\|emergent|c v v h|c v v h d|c a v h d|\bsled\b|\bcrrt\b|hemofiltration|renal replacement'),
      treatmentOffset, NULL)) AS first_acute_rrt_strict_offset,
    MIN(IF(
      treatmentOffset >= 0
      AND NOT REGEXP_CONTAINS(LOWER(treatmentString), r'for chronic renal failure|dialysis access|arteriovenous shunt|catheter')
      AND REGEXP_CONTAINS(LOWER(treatmentString), r'dialy|hemofil|renal replacement|cvvh|crrt|sled'),
      treatmentOffset, NULL)) AS first_acute_rrt_broad_offset
  FROM `{SOURCE_DATASET}.treatment`
  GROUP BY patientUnitStayID
),
io AS (
  SELECT
    patientUnitStayID,
    MIN(IF(dialysisTotal != 0, intakeOutputOffset, NULL)) AS first_nonzero_dialysis_io_offset
  FROM `{SOURCE_DATASET}.intakeoutput`
  GROUP BY patientUnitStayID
)
SELECT
  b.patientUnitStayID,
  GREATEST(COALESCE(ph.strict_chronic_ph,0),COALESCE(dx.strict_chronic_dx,0),COALESCE(tx.strict_chronic_tx,0)) AS strict_chronic_dialysis_esrd,
  GREATEST(COALESCE(ph.broad_chronic_ph,0),COALESCE(dx.broad_chronic_dx,0),COALESCE(tx.broad_chronic_tx,0)) AS broad_chronic_renal_support,
  tx.first_acute_rrt_strict_offset,
  tx.first_acute_rrt_broad_offset,
  io.first_nonzero_dialysis_io_offset
FROM base b
LEFT JOIN ph USING(patientUnitStayID)
LEFT JOIN dx USING(patientUnitStayID)
LEFT JOIN tx USING(patientUnitStayID)
LEFT JOIN io USING(patientUnitStayID);
"""
run_ddl("03_create_renal_flags_v1.sql", SQL_03)


## 04_create_creatinine_staged_v1.sql

In [ ]:
SQL_04 = f"""-- Persistent secure table: dynamic KDIGO creatinine stage at every measurement.
CREATE OR REPLACE TABLE `{TARGET_DATASET}.creatinine_staged_v1` AS
WITH refs AS (
  SELECT
    c.patientUnitStayID,
    ARRAY_AGG(STRUCT(c.labResultOffset AS ref_offset, c.creatinine AS ref_creatinine)
              ORDER BY c.labResultOffset LIMIT 1)[OFFSET(0)] AS ref
  FROM `{TARGET_DATASET}.creatinine_long_v1` c
  JOIN `{TARGET_DATASET}.base_stays_v1` b USING(patientUnitStayID)
  WHERE c.labResultOffset BETWEEN GREATEST(b.hospitalAdmitOffset,-1440) AND 360
  GROUP BY c.patientUnitStayID
),
trajectory AS (
  SELECT
    c.*,
    r.ref.ref_offset AS reference_offset,
    r.ref.ref_creatinine AS reference_creatinine,
    SAFE_DIVIDE(c.creatinine, r.ref.ref_creatinine) AS creatinine_ratio,
    MIN(c.creatinine) OVER (
      PARTITION BY c.patientUnitStayID
      ORDER BY c.labResultOffset
      RANGE BETWEEN 2880 PRECEDING AND 1 PRECEDING
    ) AS prior_min_creatinine_48h
  FROM `{TARGET_DATASET}.creatinine_long_v1` c
  JOIN refs r USING(patientUnitStayID)
),
classified AS (
  SELECT *,
    (creatinine_ratio >= 1.5
      OR (prior_min_creatinine_48h IS NOT NULL
          AND creatinine - prior_min_creatinine_48h >= 0.3)) AS aki_definition_met
  FROM trajectory
)
SELECT *,
  CASE
    WHEN creatinine_ratio >= 3.0
      OR (creatinine >= 4.0 AND aki_definition_met) THEN 3
    WHEN creatinine_ratio >= 2.0 THEN 2
    WHEN aki_definition_met THEN 1
    ELSE 0
  END AS kdigo_creatinine_stage
FROM classified;
"""
run_ddl("04_create_creatinine_staged_v1.sql", SQL_04)


## 05_create_cohort_outcome_v1.sql

In [ ]:
SQL_05 = f"""-- Persistent secure table: final cohort/outcome flags and observation adequacy.
CREATE OR REPLACE TABLE `{TARGET_DATASET}.cohort_outcome_v1` AS
WITH stage_summary AS (
  SELECT
    b.patientUnitStayID,
    b.patientHealthSystemStayID,
    b.uniquePID,
    b.hospitalID,
    b.unitDischargeOffset,
    b.unit_discharge_status,
    b.n_hospitals_for_patient,
    b.n_hospital_stays_for_patient,
    b.eligible_unit_rank_in_hospstay,
    ANY_VALUE(s.reference_offset) AS reference_offset,
    ANY_VALUE(s.reference_creatinine) AS reference_creatinine,
    MAX(IF(s.labResultOffset BETWEEN s.reference_offset AND 720, s.kdigo_creatinine_stage, NULL)) AS max_stage_by_12h,
    MIN(IF(s.labResultOffset > 720 AND s.labResultOffset <= LEAST(b.unitDischargeOffset,4320)
           AND s.kdigo_creatinine_stage >= 2, s.labResultOffset, NULL)) AS incident_stage23_offset,
    MAX(IF(s.labResultOffset > 720 AND s.labResultOffset <= LEAST(b.unitDischargeOffset,4320),
           s.kdigo_creatinine_stage, NULL)) AS max_stage_12_72h,
    MAX(IF(s.labResultOffset > 720 AND s.labResultOffset <= LEAST(b.unitDischargeOffset,4320),
           s.labResultOffset, NULL)) AS last_future_creatinine_offset,
    COUNTIF(s.labResultOffset > 2880 AND s.labResultOffset <= LEAST(b.unitDischargeOffset,4320)) AS n_creatinine_48_72h
  FROM `{TARGET_DATASET}.base_stays_v1` b
  LEFT JOIN `{TARGET_DATASET}.creatinine_staged_v1` s USING(patientUnitStayID)
  WHERE b.eligible_unit_rank_in_hospstay = 1
    AND b.n_hospitals_for_patient = 1
  GROUP BY b.patientUnitStayID,b.patientHealthSystemStayID,b.uniquePID,b.hospitalID,
           b.unitDischargeOffset,b.unit_discharge_status,b.n_hospitals_for_patient,
           b.n_hospital_stays_for_patient,b.eligible_unit_rank_in_hospstay
),
joined AS (
  SELECT s.*, r.strict_chronic_dialysis_esrd, r.broad_chronic_renal_support,
         r.first_acute_rrt_strict_offset, r.first_acute_rrt_broad_offset,
         r.first_nonzero_dialysis_io_offset,
    CASE
      WHEN s.incident_stage23_offset IS NOT NULL THEN 1
      WHEN s.unitDischargeOffset >= 4320 AND s.n_creatinine_48_72h > 0 THEN 0
      WHEN s.unitDischargeOffset > 720 AND s.unitDischargeOffset < 4320
        AND s.unit_discharge_status = 'alive'
        AND s.last_future_creatinine_offset IS NOT NULL
        AND s.last_future_creatinine_offset >= s.unitDischargeOffset - 1440 THEN 0
      ELSE NULL
    END AS outcome_creatinine_stage23,
    CASE
      WHEN s.incident_stage23_offset IS NOT NULL THEN 'positive_creatinine'
      WHEN s.unitDischargeOffset > 720 AND s.unitDischargeOffset < 4320
        AND s.unit_discharge_status = 'expired' THEN 'indeterminate_early_death'
      WHEN s.unitDischargeOffset >= 4320 AND s.n_creatinine_48_72h > 0 THEN 'negative_late_creatinine'
      WHEN s.unitDischargeOffset > 720 AND s.unitDischargeOffset < 4320
        AND s.unit_discharge_status = 'alive'
        AND s.last_future_creatinine_offset >= s.unitDischargeOffset - 1440 THEN 'negative_live_discharge_recent_creatinine'
      WHEN s.last_future_creatinine_offset IS NULL THEN 'indeterminate_no_future_creatinine'
      ELSE 'indeterminate_inadequate_followup'
    END AS observation_class
  FROM stage_summary s
  LEFT JOIN `{TARGET_DATASET}.renal_flags_v1` r USING(patientUnitStayID)
),
finalized AS (
  SELECT *,
    CASE
      WHEN outcome_creatinine_stage23 = 1 THEN 1
      WHEN first_acute_rrt_strict_offset > 720
       AND first_acute_rrt_strict_offset <= LEAST(unitDischargeOffset,4320) THEN 1
      ELSE outcome_creatinine_stage23
    END AS outcome_stage23_or_strict_rrt,
    CASE WHEN max_stage_by_12h = 1 THEN 1 ELSE 0 END AS stage1_at_prediction,
    CASE
      WHEN reference_creatinine IS NOT NULL
       AND COALESCE(strict_chronic_dialysis_esrd,0) = 0
       AND COALESCE(max_stage_by_12h,0) < 2
       AND NOT (first_acute_rrt_strict_offset IS NOT NULL AND first_acute_rrt_strict_offset <= 720)
       AND outcome_creatinine_stage23 IS NOT NULL
      THEN 1 ELSE 0
    END AS eligible_main_cohort,
    CASE
      WHEN reference_creatinine IS NOT NULL
       AND COALESCE(strict_chronic_dialysis_esrd,0) = 0
       AND COALESCE(max_stage_by_12h,0) < 2
       AND NOT (first_acute_rrt_strict_offset IS NOT NULL AND first_acute_rrt_strict_offset <= 720)
       AND (outcome_creatinine_stage23 IS NOT NULL
            OR (first_acute_rrt_strict_offset > 720
                AND first_acute_rrt_strict_offset <= LEAST(unitDischargeOffset,4320)))
      THEN 1 ELSE 0
    END AS eligible_rrt_composite_cohort
  FROM joined
)
SELECT *,
  ROW_NUMBER() OVER (
    PARTITION BY uniquePID
    ORDER BY eligible_main_cohort DESC, patientHealthSystemStayID, patientUnitStayID
  ) AS deterministic_eligible_patient_stay_rank
FROM finalized;
"""
run_ddl("05_create_cohort_outcome_v1.sql", SQL_05)


## 06_aggregate_audits.sql

In [ ]:
SQL_06 = f"""-- Shareable aggregate cohort-flow and consistency outputs.
SELECT 'base_adult_stays_12h' AS metric, COUNT(*) AS value
FROM `{TARGET_DATASET}.base_stays_v1`
UNION ALL SELECT 'first_eligible_unit_per_hospital_stay', COUNTIF(eligible_unit_rank_in_hospstay=1)
FROM `{TARGET_DATASET}.base_stays_v1`
UNION ALL SELECT 'cross_hospital_patient_stays', COUNTIF(n_hospitals_for_patient>1)
FROM `{TARGET_DATASET}.base_stays_v1`
UNION ALL SELECT 'reference_available', COUNTIF(reference_creatinine IS NOT NULL)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'strict_chronic_dialysis_esrd', COUNTIF(strict_chronic_dialysis_esrd=1)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'existing_stage23_by_12h', COUNTIF(max_stage_by_12h>=2)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'stage1_at_prediction', COUNTIF(stage1_at_prediction=1)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'incident_creatinine_stage23', COUNTIF(incident_stage23_offset IS NOT NULL)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'eligible_main_cohort', COUNTIF(eligible_main_cohort=1)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'main_events', COUNTIF(eligible_main_cohort=1 AND outcome_creatinine_stage23=1)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'main_nonevents', COUNTIF(eligible_main_cohort=1 AND outcome_creatinine_stage23=0)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'eligible_rrt_composite_cohort', COUNTIF(eligible_rrt_composite_cohort=1)
FROM `{TARGET_DATASET}.cohort_outcome_v1`
UNION ALL SELECT 'sensitivity_rrt_events', COUNTIF(eligible_rrt_composite_cohort=1 AND outcome_stage23_or_strict_rrt=1)
FROM `{TARGET_DATASET}.cohort_outcome_v1`;
"""
result_06 = run_aggregate("06_aggregate_audits.sql", SQL_06, "06_cohort_flow_aggregate.csv")


## 07_observation_class_summary.sql

In [ ]:
SQL_07 = f"""SELECT
  observation_class,
  COUNT(*) AS stays,
  COUNTIF(reference_creatinine IS NOT NULL) AS reference_available,
  COUNTIF(max_stage_by_12h >= 2) AS existing_stage23,
  COUNTIF(incident_stage23_offset IS NOT NULL) AS future_stage23
FROM `{TARGET_DATASET}.cohort_outcome_v1`
GROUP BY observation_class
ORDER BY stays DESC;
"""
result_07 = run_aggregate("07_observation_class_summary.sql", SQL_07, "07_observation_class_summary.csv")


## 08_renal_term_inventory.sql

In [ ]:
SQL_08 = f"""-- Shareable term inventory; used to audit but not tune definitions by model performance.
WITH terms AS (
  SELECT 'PASTHISTORY' AS source,
    LOWER(TRIM(CONCAT(COALESCE(pastHistoryPath,''),' | ',COALESCE(pastHistoryValue,''),' | ',COALESCE(pastHistoryValueText,'')))) AS term,
    patientUnitStayID
  FROM `{SOURCE_DATASET}.pasthistory`
  WHERE REGEXP_CONTAINS(LOWER(CONCAT(COALESCE(pastHistoryPath,''),' ',COALESCE(pastHistoryValue,''),' ',COALESCE(pastHistoryValueText,''))),
    r'renal|kidney|dialy|esrd')
  UNION ALL
  SELECT 'DIAGNOSIS', LOWER(TRIM(diagnosisString)), patientUnitStayID
  FROM `{SOURCE_DATASET}.diagnosis`
  WHERE REGEXP_CONTAINS(LOWER(diagnosisString), r'renal|kidney|dialy|esrd')
  UNION ALL
  SELECT 'TREATMENT', LOWER(TRIM(treatmentString)), patientUnitStayID
  FROM `{SOURCE_DATASET}.treatment`
  WHERE REGEXP_CONTAINS(LOWER(treatmentString), r'renal|kidney|dialy|esrd|cvvh|crrt|hemofil|sled')
)
SELECT source, term, COUNT(*) AS records, COUNT(DISTINCT patientUnitStayID) AS patient_stays
FROM terms
GROUP BY source, term
ORDER BY source, records DESC
LIMIT 500;
"""
result_08 = run_aggregate("08_renal_term_inventory.sql", SQL_08, "08_renal_term_inventory.csv")


## 09_hospital_deidentified_summary.sql

In [ ]:
SQL_09 = f"""WITH h AS (
  SELECT
    hospitalID,
    COUNTIF(eligible_main_cohort=1) AS cohort_n,
    COUNTIF(eligible_main_cohort=1 AND outcome_creatinine_stage23=1) AS event_n
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  GROUP BY hospitalID
)
SELECT 'hospital_count_with_main_cohort' AS metric, CAST(COUNTIF(cohort_n>0) AS FLOAT64) AS value FROM h
UNION ALL SELECT 'total_main_cohort', CAST(SUM(cohort_n) AS FLOAT64) FROM h
UNION ALL SELECT 'total_main_events', CAST(SUM(event_n) AS FLOAT64) FROM h
UNION ALL SELECT 'median_cohort_per_hospital', APPROX_QUANTILES(cohort_n,100)[OFFSET(50)] FROM h WHERE cohort_n>0
UNION ALL SELECT 'q25_cohort_per_hospital', APPROX_QUANTILES(cohort_n,100)[OFFSET(25)] FROM h WHERE cohort_n>0
UNION ALL SELECT 'q75_cohort_per_hospital', APPROX_QUANTILES(cohort_n,100)[OFFSET(75)] FROM h WHERE cohort_n>0
UNION ALL SELECT 'hospitals_with_at_least_20_events', CAST(COUNTIF(event_n>=20) AS FLOAT64) FROM h
UNION ALL SELECT 'hospitals_with_at_least_50_events', CAST(COUNTIF(event_n>=50) AS FLOAT64) FROM h;
"""
result_09 = run_aggregate("09_hospital_deidentified_summary.sql", SQL_09, "09_hospital_deidentified_summary.csv")


## 10_legacy_one_event_discrepancy_audit.sql

In [ ]:
SQL_10 = f"""-- Reproduces the earlier candidate definition only to explain the 3535 vs 3534 difference.
WITH adults AS (
  SELECT patientUnitStayID,hospitalID,hospitalAdmitOffset,unitDischargeOffset
  FROM `{SOURCE_DATASET}.patient`
  WHERE (CASE WHEN age='> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END)>=18
    AND unitDischargeOffset>=720
),
creat AS (
  SELECT patientUnitStayID,labResultOffset,SAFE_CAST(labResult AS FLOAT64) creatinine
  FROM `{SOURCE_DATASET}.lab`
  WHERE LOWER(TRIM(labName))='creatinine'
    AND SAFE_CAST(labResult AS FLOAT64) BETWEEN 0.1 AND 30.0
),
s AS (
  SELECT a.patientUnitStayID,a.hospitalID,
    ARRAY_AGG(IF(c.labResultOffset BETWEEN GREATEST(a.hospitalAdmitOffset,-1440) AND 360,
      STRUCT(c.labResultOffset off,c.creatinine val),NULL) IGNORE NULLS ORDER BY c.labResultOffset LIMIT 1)[SAFE_OFFSET(0)].val baseline,
    MAX(IF(c.labResultOffset BETWEEN 0 AND 720,c.creatinine,NULL)) max_0_12,
    MAX(IF(c.labResultOffset>720 AND c.labResultOffset<=4320,c.creatinine,NULL)) max_12_72
  FROM adults a LEFT JOIN creat c USING(patientUnitStayID)
  GROUP BY 1,2
),c AS (
  SELECT *,
    CASE WHEN baseline IS NULL OR max_0_12 IS NULL THEN NULL
      WHEN max_0_12>=2*baseline OR (max_0_12>=4 AND max_0_12-baseline>=0.5) THEN 1 ELSE 0 END existing,
    CASE WHEN baseline IS NULL OR max_12_72 IS NULL THEN NULL
      WHEN max_12_72>=2*baseline OR (max_12_72>=4 AND max_12_72-baseline>=0.5) THEN 1 ELSE 0 END future_event
  FROM s
)
SELECT
  COUNTIF(existing=0 AND future_event=1) AS all_candidate_events,
  COUNTIF(existing=0 AND future_event=1 AND hospitalID IS NOT NULL) AS events_with_nonnull_hospital,
  COUNTIF(existing=0 AND future_event=1 AND hospitalID IS NULL) AS events_with_null_hospital,
  COUNTIF(existing=0 AND future_event IS NOT NULL AND hospitalID IS NULL) AS analyzable_with_null_hospital
FROM c;
"""
result_10 = run_aggregate("10_legacy_one_event_discrepancy_audit.sql", SQL_10, "10_legacy_one_event_discrepancy_audit.csv")


## Güvenli paylaşım ZIP’i

In [ ]:
share_files = [
    '06_cohort_flow_aggregate.csv',
    '07_observation_class_summary.csv',
    '08_renal_term_inventory.csv',
    '09_hospital_deidentified_summary.csv',
    '10_legacy_one_event_discrepancy_audit.csv',
]
manifest=[]
for fn in share_files:
    p=os.path.join(DRIVE_OUTPUT_DIR,fn)
    b=open(p,'rb').read()
    manifest.append({'filename':fn,'size_bytes':len(b),'sha256':hashlib.sha256(b).hexdigest()})
manifest_path=os.path.join(DRIVE_OUTPUT_DIR,'MANIFEST_COHORT_OUTCOME.json')
with open(manifest_path,'w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
zip_path=os.path.join(DRIVE_OUTPUT_DIR,'AKI_V2_COHORT_OUTCOME_AGGREGATES.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for fn in share_files: z.write(os.path.join(DRIVE_OUTPUT_DIR,fn),arcname=fn)
    z.write(manifest_path,arcname='MANIFEST_COHORT_OUTCOME.json')
print('Shareable aggregate package:', zip_path)
print('Do not export or upload patient-level tables from BigQuery dataset:', TARGET_DATASET)
